#  MobileNet |


## Cell 1 — Imports

In [1]:
import matplotlib
matplotlib.use('Agg')  # Headless — saves to file instead of showing popup

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import pandas as pd
import numpy as np
import os, time, json
from glob import glob
from pathlib import Path

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_score,
    recall_score, f1_score, accuracy_score
)

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU Available: {tf.config.list_physical_devices("GPU")}')

# ─── Output directory ───────────────────────────────────────────────────────
OUTPUT_DIR = Path('/kaggle/working/dermavision_outputs')
for sub in ['curves', 'confusion_matrix', 'roc_curve', 'gradcam', 'metrics', 'model']:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f'\n✅ Output directory ready: {OUTPUT_DIR}')

2026-05-15 04:34:43.845308: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778819684.044971      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778819684.102238      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778819684.563312      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778819684.563381      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778819684.563384      23 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

✅ Output directory ready: /kaggle/working/dermavision_outputs


## Cell 2 — Data Loading & EDA

In [2]:
base_skin_dir = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000'

imageid_path_dict = {
    os.path.splitext(os.path.basename(x))[0]: x
    for x in glob(os.path.join(base_skin_dir, '*', '*.jpg'))
}

skin_df = pd.read_csv(os.path.join(base_skin_dir, 'HAM10000_metadata.csv'))
skin_df['path'] = skin_df['image_id'].map(imageid_path_dict.get)

LABEL_MAP = {
    'nv':    'Melanocytic nevi',
    'mel':   'Melanoma',
    'bkl':   'Benign keratosis-like lesions',
    'bcc':   'Basal cell carcinoma',
    'akiec': 'Actinic keratoses',
    'vasc':  'Vascular lesions',
    'df':    'Dermatofibroma'
}
skin_df['cell_type'] = skin_df['dx'].map(LABEL_MAP)
NUM_CLASSES = len(LABEL_MAP)

print(f'Total images found : {len(skin_df)}')
print(f'Classes            : {NUM_CLASSES}')
print(skin_df['cell_type'].value_counts())
skin_df.head()

Total images found : 10015
Classes            : 7
cell_type
Melanocytic nevi                 6705
Melanoma                         1113
Benign keratosis-like lesions    1099
Basal cell carcinoma              514
Actinic keratoses                 327
Vascular lesions                  142
Dermatofibroma                    115
Name: count, dtype: int64


,lesion_id,image_id,dx,dx_type,age,sex,localization,path,cell_type
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,/kaggle/input/datasets/kmader/skin-cancer-mnis...,Benign keratosis-like lesions
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,/kaggle/input/datasets/kmader/skin-cancer-mnis...,Benign keratosis-like lesions
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,/kaggle/input/datasets/kmader/skin-cancer-mnis...,Benign keratosis-like lesions
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,/kaggle/input/datasets/kmader/skin-cancer-mnis...,Benign keratosis-like lesions
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,/kaggle/input/datasets/kmader/skin-cancer-mnis...,Benign keratosis-like lesions


##  Cell 3 — Balancing + Train/Val/Test Split

In [3]:
MAX_SAMPLES = 6705
df_balanced = pd.concat([
    resample(skin_df[skin_df['dx'] == cat],
             replace=True, n_samples=MAX_SAMPLES, random_state=42)
    for cat in skin_df['dx'].unique()
])

train_df, dummy_df = train_test_split(
    df_balanced, test_size=0.30, random_state=42, stratify=df_balanced['dx'])
val_df, test_df = train_test_split(
    dummy_df, test_size=0.50, random_state=42, stratify=dummy_df['dx'])

print(f'Train : {train_df.shape}')
print(f'Val   : {val_df.shape}')
print(f'Test  : {test_df.shape}')

Train : (32854, 9)
Val   : (7040, 9)
Test  : (7041, 9)


## Cell 4 — Data Generators

In [4]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 64

# Training generator — with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

# Val / Test — only rescale
test_datagen = ImageDataGenerator(rescale=1./255)

def make_gen(datagen, df, shuffle=True):
    return datagen.flow_from_dataframe(
        df, x_col='path', y_col='cell_type',
        target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=shuffle
    )

train_gen = make_gen(train_datagen, train_df, shuffle=True)
val_gen   = make_gen(test_datagen,  val_df,   shuffle=False)
test_gen  = make_gen(test_datagen,  test_df,  shuffle=False)

CLASS_NAMES = list(train_gen.class_indices.keys())
print('Class order:', CLASS_NAMES)

Found 32854 validated image filenames belonging to 7 classes.
Found 7040 validated image filenames belonging to 7 classes.
Found 7041 validated image filenames belonging to 7 classes.
Class order: ['Actinic keratoses', 'Basal cell carcinoma', 'Benign keratosis-like lesions', 'Dermatofibroma', 'Melanocytic nevi', 'Melanoma', 'Vascular lesions']


In [5]:
# ─── Base Model ─────────────────────────────────────────────────────────────
base_model = MobileNet(weights='imagenet', include_top=False,
                       input_shape=(224, 224, 3))
base_model.trainable = False   # Freeze for transfer learning

# ─── Custom Head ────────────────────────────────────────────────────────────
x = base_model.output
x = GlobalAveragePooling2D()(x)   # GAP — needed for GradCAM++ compatibility
x = BatchNormalization()(x)
x = Dense(1024, activation='relu', name='fc1')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu', name='fc2')(x)
x = Dropout(0.3)(x)
output = Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# GradCAM++ target layer = last conv layer of MobileNet
GRADCAM_LAYER = 'conv_pw_13_relu'

print(f'Total params  : {model.count_params():,}')
print(f'GradCAM layer : {GRADCAM_LAYER}')

I0000 00:00:1778819725.235360      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778819725.241272      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Total params  : 4,550,855
GradCAM layer : conv_pw_13_relu


## Cell 6 — Training

In [6]:
EPOCHS = 15

callbacks = [
    EarlyStopping(monitor='val_loss', patience=6,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2,
                      patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(
        str(OUTPUT_DIR / 'model' / 'best_mobilenet.h5'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

history = model.fit(
    train_gen,
    steps_per_epoch  = train_gen.samples  // BATCH_SIZE,
    validation_data  = val_gen,
    validation_steps = val_gen.samples    // BATCH_SIZE,
    epochs           = EPOCHS,
    callbacks        = callbacks,
    verbose          = 1
)

# Save full model
model.save(str(OUTPUT_DIR / 'model' / 'dermavision_mobilenet_full.h5'))
print('\n✅ Model saved.')

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15


I0000 00:00:1778819736.060155      69 service.cc:152] XLA service 0x7c29a022bf00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778819736.060199      69 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778819736.060205      69 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778819736.935635      69 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-05-15 04:35:46.248212: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-15 04:35:46.389874: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1778819748.965368      69 device_co

  3/513 ━━━━━━━━━━━━━━━━━━━━ 8:06 953ms/step - accuracy: 0.1580 - loss: 3.3810

2026-05-15 04:35:59.720346: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-15 04:35:59.858524: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


...

513/513 ━━━━━━━━━━━━━━━━━━━━ 52s 102ms/step - accuracy: 0.8438 - loss: 0.4353 - val_accuracy: 0.9040 - val_loss: 0.2792 - learning_rate: 0.0010
Epoch 11/15
513/513 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8573 - loss: 0.3868
Epoch 11: val_accuracy improved from 0.90398 to 0.91136, saving model to /kaggle/working/dermavision_outputs/model/best_mobilenet.h5


513/513 ━━━━━━━━━━━━━━━━━━━━ 601s 1s/step - accuracy: 0.8573 - loss: 0.3868 - val_accuracy: 0.9114 - val_loss: 0.2588 - learning_rate: 0.0010
Epoch 12/15
  1/513 ━━━━━━━━━━━━━━━━━━━━ 33s 66ms/step - accuracy: 0.8125 - loss: 0.4362
Epoch 12: val_accuracy did not improve from 0.91136
513/513 ━━━━━━━━━━━━━━━━━━━━ 53s 103ms/step - accuracy: 0.8125 - loss: 0.4362 - val_accuracy: 0.9112 - val_loss: 0.2578 - learning_rate: 0.0010
Epoch 13/15
513/513 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8666 - loss: 0.3639
Epoch 13: val_accuracy improved from 0.91136 to 0.91534, saving model to /kaggle/working/dermavision_outputs/model/best_mobilenet.h5


513/513 ━━━━━━━━━━━━━━━━━━━━ 591s 1s/step - accuracy: 0.8666 - loss: 0.3639 - val_accuracy: 0.9153 - val_loss: 0.2408 - learning_rate: 0.0010
Epoch 14/15
  1/513 ━━━━━━━━━━━━━━━━━━━━ 34s 67ms/step - accuracy: 0.9062 - loss: 0.2514
Epoch 14: val_accuracy did not improve from 0.91534
513/513 ━━━━━━━━━━━━━━━━━━━━ 51s 100ms/step - accuracy: 0.9062 - loss: 0.2514 - val_accuracy: 0.9139 - val_loss: 0.2405 - learning_rate: 0.0010
Epoch 15/15
513/513 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8803 - loss: 0.3357
Epoch 15: val_accuracy improved from 0.91534 to 0.92088, saving model to /kaggle/working/dermavision_outputs/model/best_mobilenet.h5


513/513 ━━━━━━━━━━━━━━━━━━━━ 593s 1s/step - accuracy: 0.8803 - loss: 0.3357 - val_accuracy: 0.9209 - val_loss: 0.2269 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 15.



✅ Model saved.


## Cell 7 — Accuracy & Loss Curves (Train + Validation) → Saved

In [7]:
def plot_accuracy_loss(history, save_path):
    """Plot and save Accuracy + Loss curves (train & validation) side-by-side."""
    acc     = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss    = history.history['loss']
    val_loss= history.history['val_loss']
    epochs  = range(1, len(acc) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('DermaVision AI — MobileNet Training Curves', fontsize=16, fontweight='bold')

    # ── Accuracy ──
    axes[0].plot(epochs, acc,     'b-o', linewidth=2, markersize=4, label='Train Accuracy')
    axes[0].plot(epochs, val_acc, 'r-o', linewidth=2, markersize=4, label='Val Accuracy')
    axes[0].set_title('Accuracy Curve', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(fontsize=12); axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim([0, 1])
    # Annotate best val accuracy
    best_ep = np.argmax(val_acc)
    axes[0].annotate(f'Best: {val_acc[best_ep]:.4f}',
                     xy=(epochs[best_ep], val_acc[best_ep]),
                     xytext=(epochs[best_ep]+0.5, val_acc[best_ep]-0.05),
                     arrowprops=dict(arrowstyle='->', color='red'),
                     fontsize=10, color='red')

    # ── Loss ──
    axes[1].plot(epochs, loss,     'b-o', linewidth=2, markersize=4, label='Train Loss')
    axes[1].plot(epochs, val_loss, 'r-o', linewidth=2, markersize=4, label='Val Loss')
    axes[1].set_title('Loss Curve', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(fontsize=12); axes[1].grid(True, alpha=0.3)
    # Annotate best val loss
    best_loss_ep = np.argmin(val_loss)
    axes[1].annotate(f'Best: {val_loss[best_loss_ep]:.4f}',
                     xy=(epochs[best_loss_ep], val_loss[best_loss_ep]),
                     xytext=(epochs[best_loss_ep]+0.5, val_loss[best_loss_ep]+0.05),
                     arrowprops=dict(arrowstyle='->', color='red'),
                     fontsize=10, color='red')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'✅ Accuracy + Loss curves saved → {save_path}')

plot_accuracy_loss(history, OUTPUT_DIR / 'curves' / 'accuracy_loss_curve.png')

✅ Accuracy + Loss curves saved → /kaggle/working/dermavision_outputs/curves/accuracy_loss_curve.png


## Cell 8 — Full Metrics: Accuracy, Precision, Recall, F1, AUC, Inference/Image

In [8]:
# ─── Predict on test set ─────────────────────────────────────────────────────
test_gen.reset()

# Measure inference time
N_WARMUP = 5
dummy_batch = next(iter(test_gen))[0][:1]   # 1 image
for _ in range(N_WARMUP):                   # warm-up GPU
    model.predict(dummy_batch, verbose=0)

# Time over 100 single images
times = []
for _ in range(100):
    t0 = time.perf_counter()
    model.predict(dummy_batch, verbose=0)
    times.append(time.perf_counter() - t0)
inference_ms = np.mean(times) * 1000   # milliseconds per image

# Full test predictions
Y_prob  = model.predict(test_gen, verbose=1)   # (N, 7) softmax probabilities
y_pred  = np.argmax(Y_prob, axis=1)
y_true  = test_gen.classes

# ─── Metrics ─────────────────────────────────────────────────────────────────
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1        = f1_score(y_true, y_pred, average='weighted', zero_division=0)

# One-vs-Rest AUC (macro average)
from sklearn.preprocessing import label_binarize
y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))
auc_scores = []
for i in range(NUM_CLASSES):
    if y_true_bin[:, i].sum() > 0:
        fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], Y_prob[:, i])
        auc_scores.append(auc(fpr_i, tpr_i))
macro_auc = np.mean(auc_scores)

# ─── Print summary ───────────────────────────────────────────────────────────
metrics_dict = {
    'Accuracy'          : round(accuracy, 4),
    'Precision (W)'     : round(precision, 4),
    'Recall (W)'        : round(recall, 4),
    'F1 Score (W)'      : round(f1, 4),
    'AUC Score (Macro)' : round(macro_auc, 4),
    'Inference/Image (ms)': round(inference_ms, 3)
}

print('\n' + '='*50)
print('      📊 DermaVision AI — Test Metrics')
print('='*50)
for k, v in metrics_dict.items():
    print(f'  {k:<25} : {v}')
print('='*50)

# Save metrics to JSON
metrics_json_path = OUTPUT_DIR / 'metrics' / 'test_metrics.json'
with open(metrics_json_path, 'w') as f:
    json.dump(metrics_dict, f, indent=4)
print(f'\n✅ Metrics saved → {metrics_json_path}')

# Save per-class classification report to CSV
report_dict = classification_report(y_true, y_pred,
                                    target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()
report_csv_path = OUTPUT_DIR / 'metrics' / 'classification_report.csv'
report_df.to_csv(report_csv_path)
print(f'✅ Classification report saved → {report_csv_path}')

# Save training history to CSV
hist_df = pd.DataFrame(history.history)
hist_df.index.name = 'epoch'
hist_csv_path = OUTPUT_DIR / 'metrics' / 'training_history.csv'
hist_df.to_csv(hist_csv_path)
print(f'✅ Training history saved → {hist_csv_path}')

2026-05-15 06:03:04.561642: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-15 06:03:04.705250: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-15 06:03:04.839487: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


111/111 ━━━━━━━━━━━━━━━━━━━━ 61s 523ms/step

      📊 DermaVision AI — Test Metrics
  Accuracy                  : 0.9288
  Precision (W)             : 0.9318
  Recall (W)                : 0.9288
  F1 Score (W)              : 0.9294
  AUC Score (Macro)         : 0.994
  Inference/Image (ms)      : 78.563

✅ Metrics saved → /kaggle/working/dermavision_outputs/metrics/test_metrics.json
✅ Classification report saved → /kaggle/working/dermavision_outputs/metrics/classification_report.csv
✅ Training history saved → /kaggle/working/dermavision_outputs/metrics/training_history.csv


## Cell 9 — Confusion Matrix → Saved

In [9]:
def plot_confusion_matrix(y_true, y_pred, class_names, save_path):
    """Plot and save a styled confusion matrix."""
    cm_arr = confusion_matrix(y_true, y_pred)
    # Normalize for percentage annotation
    cm_norm = cm_arr.astype('float') / cm_arr.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(22, 8))
    fig.suptitle('Confusion Matrix — DermaVision AI (MobileNet)',
                 fontsize=16, fontweight='bold')

    # Raw counts
    sns.heatmap(cm_arr, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[0], linewidths=0.5)
    axes[0].set_title('Raw Counts', fontsize=13)
    axes[0].set_ylabel('True Label', fontsize=11)
    axes[0].set_xlabel('Predicted Label', fontsize=11)
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].tick_params(axis='y', rotation=0)

    # Normalized (%)
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
    axes[1].set_title('Normalized (Row %)', fontsize=13)
    axes[1].set_ylabel('True Label', fontsize=11)
    axes[1].set_xlabel('Predicted Label', fontsize=11)
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].tick_params(axis='y', rotation=0)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'✅ Confusion matrix saved → {save_path}')

plot_confusion_matrix(
    y_true, y_pred, CLASS_NAMES,
    OUTPUT_DIR / 'confusion_matrix' / 'confusion_matrix.png'
)

✅ Confusion matrix saved → /kaggle/working/dermavision_outputs/confusion_matrix/confusion_matrix.png


## Cell 10 — ROC Curve (per class + Macro) → Saved

In [10]:
def plot_roc_curves(y_true, Y_prob, class_names, n_classes, save_path):
    """Plot and save per-class + macro ROC curves."""
    y_true_bin = label_binarize(y_true, classes=range(n_classes))

    fpr_all, tpr_all, roc_auc = {}, {}, {}
    for i in range(n_classes):
        fpr_all[i], tpr_all[i], _ = roc_curve(y_true_bin[:, i], Y_prob[:, i])
        roc_auc[i] = auc(fpr_all[i], tpr_all[i])

    # Macro average
    all_fpr = np.unique(np.concatenate([fpr_all[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr_all[i], tpr_all[i])
    mean_tpr /= n_classes
    macro_auc_val = auc(all_fpr, mean_tpr)

    # ── Plot ──
    palette = plt.cm.get_cmap('tab10', n_classes)
    fig, ax = plt.subplots(figsize=(12, 9))

    for i, cls in enumerate(class_names):
        ax.plot(fpr_all[i], tpr_all[i], color=palette(i), lw=1.8,
                label=f'{cls}  (AUC = {roc_auc[i]:.3f})')

    ax.plot(all_fpr, mean_tpr, 'k--', lw=2.5,
            label=f'Macro Avg  (AUC = {macro_auc_val:.3f})')
    ax.plot([0,1],[0,1], 'gray', lw=1, linestyle=':', label='Random Classifier')

    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.set_xlabel('False Positive Rate', fontsize=13)
    ax.set_ylabel('True Positive Rate', fontsize=13)
    ax.set_title('ROC Curves — DermaVision AI (MobileNet)',
                 fontsize=15, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9.5)
    ax.grid(True, alpha=0.25)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

    # Save per-class AUC to CSV
    auc_df = pd.DataFrame({
        'Class': class_names,
        'AUC'  : [round(roc_auc[i], 4) for i in range(n_classes)]
    })
    auc_df.loc[len(auc_df)] = ['Macro Average', round(macro_auc_val, 4)]
    auc_csv = save_path.parent / 'per_class_auc.csv'
    auc_df.to_csv(auc_csv, index=False)
    print(f'✅ ROC curve saved → {save_path}')
    print(f'✅ Per-class AUC saved → {auc_csv}')

plot_roc_curves(
    y_true, Y_prob, CLASS_NAMES, NUM_CLASSES,
    OUTPUT_DIR / 'roc_curve' / 'roc_curve.png'
)

/tmp/ipykernel_23/2051633983.py:19: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  palette = plt.cm.get_cmap('tab10', n_classes)


✅ ROC curve saved → /kaggle/working/dermavision_outputs/roc_curve/roc_curve.png
✅ Per-class AUC saved → /kaggle/working/dermavision_outputs/roc_curve/per_class_auc.csv


## Cell 11 — GradCAM++ (Explainable AI) → Saved

In [ ]:
# ─── GradCAM++ Implementation ────────────────────────────────────────────────

def get_gradcam_pp(model, img_array, layer_name, class_idx=None):
    """
    Compute GradCAM++ heatmap.
    Returns: heatmap (H, W) in [0, 1]
    """
    grad_model = Model(
        inputs  = model.input,
        outputs = [model.get_layer(layer_name).output, model.output]
    )

    with tf.GradientTape() as tape2:
        with tf.GradientTape() as tape1:
            with tf.GradientTape() as tape0:
                conv_out, preds = grad_model(img_array, training=False)
                if class_idx is None:
                    class_idx = tf.argmax(preds[0])
                score = preds[:, class_idx]
            grads1 = tape0.gradient(score, conv_out)   # 1st derivative
        grads2 = tape1.gradient(grads1, conv_out)       # 2nd derivative
    grads3 = tape2.gradient(grads2, conv_out)           # 3rd derivative

    # GradCAM++ alpha weights
    denom       = 2.0 * grads2 + tf.reduce_sum(grads3 * conv_out,
                                                axis=(1, 2), keepdims=True)
    denom       = tf.where(denom == 0, tf.ones_like(denom), denom)
    alphas      = grads2 / denom

    relu_grads  = tf.nn.relu(score) * grads1
    weights     = tf.reduce_sum(alphas * relu_grads, axis=(1, 2))

    conv_out_np = conv_out[0].numpy()
    weights_np  = weights[0].numpy()

    cam = np.zeros(conv_out_np.shape[:2], dtype=np.float32)
    for i, w in enumerate(weights_np):
        cam += w * conv_out_np[:, :, i]

    cam = np.maximum(cam, 0)
    cam = cam / (cam.max() + 1e-8)
    return cam, int(class_idx)


def overlay_gradcam(img_orig, heatmap, alpha=0.45):
    """Overlay GradCAM++ heatmap on the original image."""
    heatmap_resized = np.uint8(255 * heatmap)
    heatmap_colored = cm.jet(heatmap_resized)[:, :, :3]  # RGBA → RGB
    heatmap_colored = np.uint8(heatmap_colored * 255)

    # Resize heatmap to match original image
    from PIL import Image as PILImage
    heatmap_pil = PILImage.fromarray(heatmap_colored).resize(
        (img_orig.shape[1], img_orig.shape[0]), PILImage.LANCZOS
    )
    heatmap_resized = np.array(heatmap_pil)

    overlay = (1 - alpha) * img_orig + alpha * heatmap_resized
    return np.uint8(np.clip(overlay, 0, 255))


def run_gradcam_on_test(model, test_gen, class_names, layer_name,
                         save_dir, n_per_class=2):
    """
    Run GradCAM++ on n_per_class images for every class.
    Saves individual images + a summary grid.
    """
    save_dir = Path(save_dir)
    class_idx_map = {v: k for k, v in test_gen.class_indices.items()}
    n_classes = len(class_names)

    # Collect indices per class
    all_indices = np.arange(len(test_gen.filenames))
    class_sample_indices = {}
    for cls_idx in range(n_classes):
        cls_mask = test_gen.classes == cls_idx
        pool = all_indices[cls_mask]
        chosen = pool[:n_per_class] if len(pool) >= n_per_class else pool
        class_sample_indices[cls_idx] = chosen

    # ── Build summary grid ──
    n_cols  = n_per_class * 3   # original | heatmap | overlay
    n_rows  = n_classes
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 3, n_rows * 3.2))
    fig.suptitle('GradCAM++ — DermaVision AI (MobileNet)\n'
                 'Original | Heatmap | Overlay  (per class)',
                 fontsize=14, fontweight='bold')

    for row_idx, (cls_idx, sample_idxs) in enumerate(class_sample_indices.items()):
        cls_name = class_idx_map[cls_idx]

        for col_group, sample_idx in enumerate(sample_idxs):
            # Load image
            img_path = test_gen.filenames[sample_idx]
            img_pil  = load_img(img_path, target_size=(224, 224))
            img_np   = img_to_array(img_pil)        # uint8 [0,255]
            img_inp  = img_np / 255.0               # float [0,1]
            img_inp_exp = np.expand_dims(img_inp, 0).astype('float32')

            # GradCAM++
            heatmap, pred_idx = get_gradcam_pp(model, img_inp_exp,
                                               layer_name, cls_idx)
            overlay = overlay_gradcam(img_np, heatmap)

            # Heatmap colored (for display)
            hm_colored = np.uint8(cm.jet(
                np.uint8(255 * heatmap))[:, :, :3] * 255)

            # Save individual images
            prefix = save_dir / f'cls{cls_idx}_{cls_name.replace(" ","_")}_img{col_group}'
            plt.imsave(str(prefix) + '_original.png', img_np.astype(np.uint8))
            plt.imsave(str(prefix) + '_heatmap.png',  hm_colored)
            plt.imsave(str(prefix) + '_overlay.png',  overlay)

            # Fill grid
            base_col = col_group * 3
            for ax in axes[row_idx][base_col:base_col+3]:
                ax.axis('off')

            axes[row_idx][base_col].imshow(img_np.astype(np.uint8))
            axes[row_idx][base_col].set_title('Original' if row_idx==0 else '', fontsize=8)

            axes[row_idx][base_col+1].imshow(hm_colored)
            axes[row_idx][base_col+1].set_title('GradCAM++' if row_idx==0 else '', fontsize=8)

            axes[row_idx][base_col+2].imshow(overlay)
            axes[row_idx][base_col+2].set_title('Overlay' if row_idx==0 else '', fontsize=8)

        # Row label
        axes[row_idx][0].set_ylabel(cls_name, fontsize=9,
                                    fontweight='bold', labelpad=5)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    grid_path = save_dir / 'gradcam_summary_grid.png'
    plt.savefig(str(grid_path), dpi=130, bbox_inches='tight')
    plt.close()
    print(f'✅ GradCAM++ summary grid saved → {grid_path}')
    print(f'✅ Individual GradCAM++ images saved in → {save_dir}')


# ─── Run ─────────────────────────────────────────────────────────────────────
test_gen.reset()
run_gradcam_on_test(
    model      = model,
    test_gen   = test_gen,
    class_names= CLASS_NAMES,
    layer_name = GRADCAM_LAYER,
    save_dir   = OUTPUT_DIR / 'gradcam',
    n_per_class= 2          # 2 sample images per class (adjust freely)
)

## Cell 12 — Final Summary

In [12]:
print('\n' + '='*60)
print('  ✅  DermaVision AI — All Outputs Saved')
print('='*60)

saved_files = list(OUTPUT_DIR.rglob('*'))
for f in sorted(saved_files):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f'  📄 {str(f.relative_to(OUTPUT_DIR)):55s}  ({size_kb:7.1f} KB)')

print('='*60)
print(f'\n  📊 Final Test Metrics:')
for k, v in metrics_dict.items():
    print(f'      {k:<28}: {v}')
print('='*60)


  ✅  DermaVision AI — All Outputs Saved
  📄 confusion_matrix/confusion_matrix.png                    (  228.2 KB)
  📄 curves/accuracy_loss_curve.png                           (  115.6 KB)
  📄 gradcam/gradcam_summary_grid.png                         (  145.5 KB)
  📄 metrics/classification_report.csv                        (    0.8 KB)
  📄 metrics/test_metrics.json                                (    0.2 KB)
  📄 metrics/training_history.csv                             (    1.5 KB)
  📄 model/best_mobilenet.h5                                  (28382.6 KB)
  📄 model/dermavision_mobilenet_full.h5                      (28382.6 KB)
  📄 roc_curve/per_class_auc.csv                              (    0.2 KB)
  📄 roc_curve/roc_curve.png                                  (  146.8 KB)

  📊 Final Test Metrics:
      Accuracy                    : 0.9288
      Precision (W)               : 0.9318
      Recall (W)                  : 0.9288
      F1 Score (W)                : 0.9294
      AUC Score (Macro